In [55]:


from dotenv import load_dotenv
import os
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict

os.environ["USER_AGEENT"] = "aO"

# Load and chunk contents of the blog
# loader = WebBaseLoader(
#     web_paths=("https://github.com/abhishekabhi779/RAG/blob/main/Drug%20data",),
#     bs_kwargs=dict(
#         parse_only=bs4.SoupStrainer(
#             class_=("post-content", "post-title", "post-header")
#         )
#     ),
# )
# docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

In [56]:
all_splits

[Document(metadata={}, page_content='# OptiClear-X: Phase III Clinical Trial Results\nPublished: January 2024\nPrincipal Investigators: Dr. Sarah Chen, Dr. Michael Roberts\nInstitution: Vision Research Institute\n\n## Drug Overview\nOptiClear-X (ocumapril) is a novel small-molecule drug targeting axial elongation in myopic patients. The drug works by inhibiting TGF-β signaling in the scleral tissue, preventing excessive eye growth associated with myopia progression.\n\n## Chemical Properties\n- Formula: C23H29N3O4\n- Half-life: 14.2 hours\n- Bioavailability: 78% when administered as eye drops\n- Storage: Stable at 2-8°C for 24 months\n\n## Clinical Trial Design\n- Double-blind, randomized, placebo-controlled study\n- Duration: 24 months\n- Participants: 2,842 patients (ages 12-25)\n- Treatment protocol: One drop per eye, twice daily\n- Primary endpoint: Change in spherical equivalent refraction\n- Secondary endpoint: Axial length change\n\n## Results'),
 Document(metadata={}, page_cont

In [57]:
import requests

url = "https://raw.githubusercontent.com/abhishekabhi779/RAG/main/Drug%20data"
response = requests.get(url)
if response.status_code == 200:
    content = response.text
    docs = [Document(page_content=content)]
    all_splits = text_splitter.split_documents(docs)
   


Step 3: Generating Embeddings with Ollama

In [58]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

local_embeddings = OllamaEmbeddings(model="all-minilm")
vectorstore = Chroma.from_documents(documents=all_splits, embedding=local_embeddings)

In [59]:
question = "what are the side effects?"
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retrieved_docs = retriever.invoke(question)

In [60]:
retrieved_docs

[Document(id='31476d04-7a25-44f2-b798-abd3e4d11f99', metadata={}, page_content='## Long-term Follow-up Data\n24-month outcomes:\n- Maintained improvement: 91%\n- Regression rate: 3%\n- Quality of life improvement: 87%\n- Patient satisfaction: 8.4/10\n\n## Regulatory Status\n- FDA Fast Track Designation granted\n- EMA Priority Review status\n- Patent protection until 2041\n- Manufacturing approval in 12 countries\n\n## Current Research\nOngoing studies:\n- Prevention trial in high-risk children\n- Combination therapy with atropine\n- Extended release formulation development\n- Geographic variations in efficacy'),
 Document(id='34a74ec9-9653-425b-8d22-4fd810bdcfff', metadata={}, page_content='## Long-term Follow-up Data\n24-month outcomes:\n- Maintained improvement: 91%\n- Regression rate: 3%\n- Quality of life improvement: 87%\n- Patient satisfaction: 8.4/10\n\n## Regulatory Status\n- FDA Fast Track Designation granted\n- EMA Priority Review status\n- Patent protection until 2041\n- Man

In [61]:
context = ' '.join([doc.page_content for doc in retrieved_docs])
context

'## Long-term Follow-up Data\n24-month outcomes:\n- Maintained improvement: 91%\n- Regression rate: 3%\n- Quality of life improvement: 87%\n- Patient satisfaction: 8.4/10\n\n## Regulatory Status\n- FDA Fast Track Designation granted\n- EMA Priority Review status\n- Patent protection until 2041\n- Manufacturing approval in 12 countries\n\n## Current Research\nOngoing studies:\n- Prevention trial in high-risk children\n- Combination therapy with atropine\n- Extended release formulation development\n- Geographic variations in efficacy ## Long-term Follow-up Data\n24-month outcomes:\n- Maintained improvement: 91%\n- Regression rate: 3%\n- Quality of life improvement: 87%\n- Patient satisfaction: 8.4/10\n\n## Regulatory Status\n- FDA Fast Track Designation granted\n- EMA Priority Review status\n- Patent protection until 2041\n- Manufacturing approval in 12 countries\n\n## Current Research\nOngoing studies:\n- Prevention trial in high-risk children\n- Combination therapy with atropine\n- Ext

LLM Response

In [62]:
from langchain_ollama.llms import OllamaLLM

In [63]:
llm = OllamaLLM(model="llama3.2:3b")

response = llm.invoke(f"""Answer the question from the context.be very clear and concise.: 
                      question: {question} context: {context}""")

In [64]:
print(response)

There are no explicit side effects listed in the provided context.
